# Train and compare BDT feature sets across pT bins

Edit `multiModelsBDT_config.json` to choose the named feature lists and `PT_BINS`. Every feature-set/bin combination is trained with the same split seed.


In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import auc, roc_curve
from hipe4ml import analysis_utils
import singleModelBDT
from utils.hist_utils import createGeomPtBins2

CONFIG_PATH = Path('multiModelsBDT_config.json')
with CONFIG_PATH.open() as config_file:
    config = json.load(config_file)

FEATURE_SETS = config['FEATURE_SETS']
PT_BINS = [tuple(pt_bin) for pt_bin in config['PT_BINS']]
OUTPUT_ROOT = Path(config['OUT_DIR'])
TRAINING = config.get('TRAINING', {})
CLASS_LABELS = ['background', 'prompt', 'nonprompt']
if not FEATURE_SETS or not PT_BINS:
    raise ValueError('FEATURE_SETS and PT_BINS must both be non-empty')

# from importlib import reload
# reload(utils.hist_utils)
# reload(singleModelBDT)


## Train every configured combination


In [ ]:
training_results = {}
for model_name, features in FEATURE_SETS.items():
    for pt_bin in PT_BINS:
        pt_tag = f"pt{pt_bin[0]}_{pt_bin[1]}"
        run_dir = OUTPUT_ROOT / model_name / pt_tag
        print(f'Processing {model_name} for pT {pt_bin[0]}–{pt_bin[1]} in {run_dir}')
        training_results[(model_name, pt_bin)] = singleModelBDT.runSingleModel(
            features=features,
            pt_bin=pt_bin,
            model_name=model_name,
            out_dir=run_dir,
            data_file=config['DATA_FILE'],
            promptmc_file=config['PROMPTMC_FILE'],
            nonpromptmc_file=config['NONPROMPTMC_FILE'],
            n_estimators=tuple(TRAINING.get('N_ESTIMATORS', [20, 1000])),
            max_depth=tuple(TRAINING.get('MAX_DEPTH', [2, 6])),
            learning_rate=tuple(TRAINING.get('LEARNING_RATE', [0.01, 0.1])),
            n_jobs=TRAINING.get('N_JOBS', 10),
            timeout=TRAINING.get('TIMEOUT', 60),
            n_trials=TRAINING.get('N_TRIALS', 100),
            random_state=TRAINING.get('RANDOM_STATE', 42),
            plotFigures=False,
            use_previous_model=True
        )

print(f'Trained {len(training_results)} model/bin configurations')


## Compare ROC curves


In [ ]:
comparison_results = {}
for (model_name, pt_bin), result in training_results.items():
    with np.load(result['comparison_file']) as saved:
        comparison_results[(model_name, pt_bin)] = {
            'y_test': saved['y_test'].copy(),
            'y_score': saved['y_score_test'].copy(),
        }

comparison_dir = OUTPUT_ROOT / 'comparison'
comparison_dir.mkdir(parents=True, exist_ok=True)
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharex=True, sharey=True)
for class_id, ax in enumerate(axes):
    for (model_name, pt_bin), result in comparison_results.items():
        y_binary = (result['y_test'] == class_id).astype(int)
        fpr, tpr, _ = roc_curve(y_binary, result['y_score'][:, class_id])
        curve_auc = auc(fpr, tpr)
        label = f"{model_name}, pT {pt_bin[0]}–{pt_bin[1]} (AUC={curve_auc:.3f})"
        ax.plot(fpr, tpr, label=label)
    ax.plot([0, 1], [0, 1], 'k--', linewidth=1)
    ax.set(title=f'{CLASS_LABELS[class_id]} one-vs-rest', xlabel='False positive rate', ylabel='True positive rate')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.02)
    ax.grid(alpha=0.3)
    ax.legend(fontsize='x-small')
fig.tight_layout()
fig.savefig(comparison_dir / 'model_comparison_roc.pdf', bbox_inches='tight')
plt.show()
plt.close(fig)


In [ ]:
y = createGeomPtBins2(1, 30, 25, 1)
x = range(26)
plt.plot(x, y, 'o-')

## Compare pT efficiencies

Each figure uses the same pT and efficiency axes. Solid curves show prompt efficiency; dashed curves show nonprompt efficiency. Colors identify feature sets.


In [ ]:
NnonpCuts = 6
bkgCut = 0.8

pt_min = 1
pt_max = 30
pt_nBins = 30
beta = 1
pt_axis = createGeomPtBins2(pt_min, pt_max, pt_nBins, beta)

efficiency_cuts = np.round(np.linspace(0, 0.95, NnonpCuts), 2)
pt_bin_styles = ['o', 's', '^', 'D', 'v', '*', 'x', '+']
myPallete = ['maroon', 'navy', 'darkorange', 'darkgreen', 'darkmagenta']
model_colors = {name: myPallete[index % len(myPallete)] for index, name in enumerate(FEATURE_SETS)}
efficiencies_by_model = {}
observed_pt_ranges = []

# Recalculate from the saved test predictions so older NPZ efficiency arrays
# with a fixed pT upper limit cannot truncate the model-bin curves.
for model_name in FEATURE_SETS:
    for pt_bin in PT_BINS:
        result = training_results[(model_name, pt_bin)]
        with np.load(result['comparison_file']) as saved:
            y_test = saved['y_test'].copy()
            y_score = saved['y_score_test'].copy()
            pt_values = saved['pt_test'].copy()

        if pt_values.size == 0:
            continue
        pt_max_available = min(float(pt_bin[1]), float(np.max(pt_values))) # From all candidates
        if pt_max_available <= float(np.min(pt_values)):
            continue
        observed_pt_ranges.append((float(np.min(pt_values)), pt_max))

        efficiency_curves = []
        for nonprompt_cut in efficiency_cuts:
            curve = analysis_utils.bdt_efficiency_vs_var(
                y_test, y_score, pt_values, nonprompt_cut, bkgCut,
                n_points=20, var_max=pt_max,
            )
            efficiency_curves.append(curve)
        efficiencies_by_model[(model_name, pt_bin)] = efficiency_curves

if not observed_pt_ranges:
    raise ValueError('No saved test candidates are available for efficiency comparisons')
pt_plot_min = min(low for low, high in observed_pt_ranges)
pt_plot_max = max(high for low, high in observed_pt_ranges)

for efficiency_index, nonprompt_cut in enumerate(efficiency_cuts):
    fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharex=True, sharey=True)
    for model_name in FEATURE_SETS:
        for pt_index, pt_bin in enumerate(PT_BINS):
            curves = efficiencies_by_model.get((model_name, pt_bin))
            if curves is None:
                continue
            centers, prompt_eff, prompt_err, nonprompt_eff, nonprompt_err = curves[efficiency_index]
            in_model_range = (centers >= pt_bin[0]) & (centers < pt_bin[1])
            if not np.any(in_model_range):
                continue

            label = f"{model_name}, pT {pt_bin[0]}–{pt_bin[1]}"
            style = pt_bin_styles[pt_index % len(pt_bin_styles)]
            axes[0].errorbar(
                centers[in_model_range], prompt_eff[in_model_range],
                yerr=prompt_err[in_model_range], color=model_colors[model_name],
                linestyle='-', marker=style, capsize=2, label=label,
            )
            axes[1].errorbar(
                centers[in_model_range], nonprompt_eff[in_model_range],
                yerr=nonprompt_err[in_model_range], color=model_colors[model_name],
                linestyle='-', marker=style, capsize=2, label=label,
            )

    for ax, title in zip(axes, ('Prompt efficiency', 'Nonprompt efficiency')):
        ax.set(title=title, xlabel=r'$p_{\mathrm{T}}^{\mathrm{J}/\psi}$ (GeV/$c$)', ylabel='Efficiency')
        ax.set_xlim(pt_plot_min, pt_plot_max)
        ax.set_ylim(0, 1.05)
        ax.grid(alpha=0.3)
        ax.legend(title='Feature set, pT model bin', fontsize='small')

    fig.suptitle(f'Efficiency comparison across pT-model bins; nonprompt score cut {nonprompt_cut:.2f}, bkg cut {bkgCut}')
    fig.tight_layout()
    fig.savefig(
        comparison_dir / f'efficiency_comparison_all_pt_bins_nonpromptCut{nonprompt_cut:.2f}.pdf',
        bbox_inches='tight',
    )
    plt.show()
    plt.close(fig)
